# Serving Notebook - MLflow Model Serving

This notebook demonstrates:
- Training and saving a model with MLflow
- Serving the model via MLflow's REST API
- Sending prediction requests to the served model

**Dataset:** California Housing (sklearn built-in)  
**Model:** Decision Tree Regressor  
**Serving:** MLflow REST API on localhost

In [4]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
import json
import requests
mlflow.set_tracking_uri("file:./mlruns")

## 1. Train and Log the Model

In [5]:
# Load data
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Set experiment
mlflow.set_experiment("california-housing-serving")

with mlflow.start_run(run_name="serving-decision-tree") as run:
    # Train
    model = DecisionTreeRegressor(max_depth=12, min_samples_split=5, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Log
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_split", 5)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2_score", r2)
    mlflow.sklearn.log_model(model, "decision-tree-model")
    
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"MSE: {mse:.4f}, R2: {r2:.4f}")

/Users/harsha/Desktop/MLOps Labs/MLOps_Labs/lab4/mlflow_lab1/mlflow_lab1_env/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/14 22:04:44 INFO mlflow.tracking.fluent: Experiment with name 'california-housing-serving' does not exist. Creating a new experiment.
2026/03/14 22:04:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 22:04:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary c

Run ID: 941e02b53e8440f7b2bba8dd3d7557f2
MSE: 0.4279, R2: 0.6734


## 2. Serve the Model

Open a **new terminal** in VSCode, activate your virtual environment, and run:

```bash
mlflow models serve --env-manager=local -m runs:/<RUN_ID>/decision-tree-model -p 5001
```

Replace `<RUN_ID>` with the run ID printed above.

Alternatively, you can find the run ID from the `mlruns/` directory.

In [6]:
# Print the serve command with the actual run ID
print(f"Run this command in a new terminal:")
print(f"\nmlflow models serve --env-manager=local -m runs:/{run_id}/decision-tree-model -p 5001")

Run this command in a new terminal:

mlflow models serve --env-manager=local -m runs:/941e02b53e8440f7b2bba8dd3d7557f2/decision-tree-model -p 5001


## 3. Send Prediction Requests

Once the model is being served (after running the command above), run the cells below to send prediction requests.

In [7]:
# Define the serving URL
url = 'http://127.0.0.1:5002/invocations'

# Prepare sample data for prediction (3 samples from test set)
sample_data = X_test.head(3)
print("Sample input data:")
print(sample_data)
print(f"\nActual values: {y_test[:3]}")

Sample input data:
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
20046  1.6812      25.0  4.192201   1.022284      1392.0  3.877437     36.06   
3024   2.5313      30.0  5.039384   1.193493      1565.0  2.679795     35.14   
15663  3.4801      52.0  3.977155   1.185877      1310.0  1.360332     37.80   

       Longitude  
20046    -119.01  
3024     -119.46  
15663    -122.44  

Actual values: [0.477   0.458   5.00001]


In [8]:
# Send prediction request using dataframe_split format
headers = {'Content-Type': 'application/json'}

# Format data as dataframe_split (recommended format)
payload = {
    "dataframe_split": {
        "columns": list(sample_data.columns),
        "data": sample_data.values.tolist()
    }
}

try:
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        predictions = response.json()
        print(f"Predictions from served model: {predictions}")
    else:
        print(f"Error {response.status_code}: {response.text}")
        print("\nMake sure the model server is running in another terminal!")
except requests.exceptions.ConnectionError:
    print("Connection refused. Make sure the model server is running.")
    print(f"Run: mlflow models serve --env-manager=local -m runs:/{run_id}/decision-tree-model -p 5001")

Predictions from served model: {'predictions': [0.5496888888888891, 0.8022758620689656, 5.000010000000001]}


In [9]:
# Compare served model predictions with local predictions
local_predictions = model.predict(sample_data)
print(f"Local model predictions: {local_predictions.tolist()}")
print(f"\nNote: Served model predictions should match local predictions exactly.")

Local model predictions: [0.5496888888888891, 0.8022758620689656, 5.000010000000001]

Note: Served model predictions should match local predictions exactly.
